# ML-01 — Algorithms: Hands-on Starter Notebook
**HBE_2206 | Hazem Elshafey**

This notebook is a runnable companion to the ML-01 guide. It demonstrates regression, classification, and K-Means with scikit-learn's built-in datasets. Run from top to bottom.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_breast_cancer, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, ConfusionMatrixDisplay, silhouette_score
from sklearn.cluster import KMeans

RANDOM_STATE = 42

## 1. Regression — baseline vs Linear Regression vs Random Forest

We use the built-in diabetes dataset. The target is a quantitative disease-progression measure.

In [ ]:
diabetes = load_diabetes(as_frame=True)
X_reg, y_reg = diabetes.data, diabetes.target
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

regressors = {
    'Dummy baseline': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
}

reg_results = []
for name, model in regressors.items():
    model.fit(Xr_train, yr_train)
    pred = model.predict(Xr_test)
    reg_results.append({
        'Model': name,
        'MAE': mean_absolute_error(yr_test, pred),
        'RMSE': mean_squared_error(yr_test, pred) ** 0.5,
        'R2': r2_score(yr_test, pred)
    })

pd.DataFrame(reg_results).sort_values('RMSE')

**Think:** Why is the dummy model an important comparison? Does the model with the lowest RMSE also have the highest R² here?

## 2. Classification — Logistic Regression and Random Forest

We use the built-in breast-cancer dataset for a binary classification demonstration. This is a learning example, not a clinical tool.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X_cls, y_cls = cancer.data, cancer.target
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.2, stratify=y_cls, random_state=RANDOM_STATE
)

classifiers = {
    'Dummy baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
}

for name, model in classifiers.items():
    model.fit(Xc_train, yc_train)
    pred = model.predict(Xc_test)
    print(f'\\n{name}')
    print(classification_report(yc_test, pred, target_names=cancer.target_names))

In [ ]:
best_classifier = classifiers['Logistic Regression']
ConfusionMatrixDisplay.from_estimator(best_classifier, Xc_test, yc_test, display_labels=cancer.target_names)
plt.title('Logistic Regression — Confusion Matrix')
plt.show()

**Think:** Which class has lower recall? What could be the consequence of a false negative in a high-stakes setting? Why should a threshold be chosen with the application in mind?

## 3. Clustering — K-Means and the choice of K

The synthetic dataset below has three generated groups. In real projects, the number and shape of clusters are not known in advance.

In [ ]:
X_cluster, _ = make_blobs(n_samples=500, centers=3, cluster_std=0.75, random_state=RANDOM_STATE)
scores = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_cluster)
    scores.append({'k': k, 'inertia': km.inertia_, 'silhouette': silhouette_score(X_cluster, labels)})

cluster_results = pd.DataFrame(scores)
cluster_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cluster_results['k'], cluster_results['inertia'], marker='o')
axes[0].set(title='Elbow / Inertia', xlabel='Number of clusters (K)', ylabel='Inertia')
axes[1].plot(cluster_results['k'], cluster_results['silhouette'], marker='o')
axes[1].set(title='Silhouette Score', xlabel='Number of clusters (K)', ylabel='Score')
plt.tight_layout()
plt.show()

## 4. Your practice tasks

1. Change the train/test split and observe how the scores change.
2. Add a Decision Tree or KNN classifier and compare it with the baseline.
3. Explain why scaling is included in the Logistic Regression pipeline.
4. Try K-Means with a different random state and compare the result.
5. Write down one limitation of each experiment.

## Important notes
- These datasets are included with scikit-learn; no external downloads are required.
- Metrics from a single split can vary. Use cross-validation for model selection in a real project.
- The clustering dataset is synthetic and intentionally simple.
- This notebook is a starter companion. Run all cells in Colab/Jupyter and verify outputs before treating it as validated.